# Module 4 Lab 02 - Pandas Cleaning and Transformation Workflow

This notebook covers DataFrame inspection, type conversion, null handling, duplicate checks, filtering, grouping, pivoting, joining public indicator data, and preparing analytical outputs.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas", "numpy", "requests", "openpyxl"])

from pathlib import Path
import numpy as np
import pandas as pd
import requests

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
SAMPLE_CSV = DATA_DIR / "m4_raw_financial_transactions_sample.csv"

def load_sample_transactions():
    if SAMPLE_CSV.exists():
        return pd.read_csv(SAMPLE_CSV)
    print("Local CSV not found; using built-in fallback sample rows.")
    return pd.DataFrame([
        {"SourceSystem": "CoreBanking", "TransactionReference": "M4-0001", "TransactionDateText": "2026-06-01", "InstitutionCode": "MCB", "CounterpartyName": "Maseru Commercial Bank", "CurrencyCode": "LSL", "AmountText": "15000.00", "TransactionType": "Deposit", "Channel": "Branch", "LoadBatch": "BATCH-202606"},
        {"SourceSystem": "Payments", "TransactionReference": "M4-0003", "TransactionDateText": "2026/06/02", "InstitutionCode": "CPS", "CounterpartyName": "Cape Payment Services", "CurrencyCode": "ZAR", "AmountText": "25000", "TransactionType": "Transfer", "Channel": "Clearing", "LoadBatch": "BATCH-202606"},
        {"SourceSystem": "Treasury", "TransactionReference": "M4-0004", "TransactionDateText": "2026-06-03", "InstitutionCode": "CBL", "CounterpartyName": "Central Bank of Lesotho", "CurrencyCode": "USD", "AmountText": "100000.00", "TransactionType": "FX Settlement", "Channel": "SWIFT", "LoadBatch": "BATCH-202606"},
        {"SourceSystem": "CoreBanking", "TransactionReference": "M4-0006", "TransactionDateText": None, "InstitutionCode": "MCB", "CounterpartyName": "Maseru Commercial Bank", "CurrencyCode": "LSL", "AmountText": "9100.00", "TransactionType": "Deposit", "Channel": "Branch", "LoadBatch": "BATCH-202606"},
        {"SourceSystem": "Treasury", "TransactionReference": "M4-0010", "TransactionDateText": "2026-06-08", "InstitutionCode": "CBL", "CounterpartyName": "Central Bank of Lesotho", "CurrencyCode": "usd", "AmountText": "bad_amount", "TransactionType": "FX Settlement", "Channel": "SWIFT", "LoadBatch": "BATCH-202606"},
    ])

In [ ]:
# Load the local training CSV fallback.
# This is the same shape as the SQL Server m4.RawFinancialTransactions table.
raw_df = load_sample_transactions()
raw_df.insert(0, "RawTransactionID", range(1, len(raw_df) + 1))

display(raw_df.head())
raw_df.info()

In [ ]:
# Initial quality profile: null counts, duplicate references, and raw currency values.
quality_profile = pd.DataFrame({
    "column": raw_df.columns,
    "null_count": raw_df.isna().sum().to_numpy(),
    "null_percent": (raw_df.isna().mean() * 100).round(2).to_numpy(),
})

display(quality_profile)
print("Duplicate references:", raw_df["TransactionReference"].duplicated().sum())
print("Raw currency values:", sorted(raw_df["CurrencyCode"].dropna().unique()))

In [ ]:
# Cleaning and type conversion workflow.
FX_RATES_TO_LSL = {"LSL": 1.0, "ZAR": 1.0, "USD": 18.25, "EUR": 19.75, "GBP": 23.10}

df = raw_df.copy()
df["TransactionDate"] = pd.to_datetime(df["TransactionDateText"], format="mixed", errors="coerce", dayfirst=False)
df["CurrencyCode"] = df["CurrencyCode"].astype("string").str.strip().str.upper()
df["Amount"] = pd.to_numeric(df["AmountText"], errors="coerce")
df["InstitutionCode"] = df["InstitutionCode"].astype("string").str.strip().str.upper()
df["CounterpartyName"] = df["CounterpartyName"].fillna("Unknown Counterparty")
df["TransactionType"] = df["TransactionType"].fillna("Unclassified")
df["Channel"] = df["Channel"].fillna("Unknown")

required_columns = ["TransactionDate", "InstitutionCode", "CurrencyCode", "Amount", "TransactionType", "Channel"]
valid_mask = df[required_columns].notna().all(axis=1)
valid_mask &= df["CurrencyCode"].isin(FX_RATES_TO_LSL)
valid_mask &= df["Amount"] >= 0

clean_df = df.loc[valid_mask].copy()
rejected_df = df.loc[~valid_mask].copy()

clean_df["FxRateToLSL"] = clean_df["CurrencyCode"].map(FX_RATES_TO_LSL)
clean_df["AmountLSL"] = (clean_df["Amount"] * clean_df["FxRateToLSL"]).round(2)
clean_df["AmountBand"] = pd.cut(
    clean_df["Amount"],
    bins=[-0.01, 10000, 50000, np.inf],
    labels=["Small", "Medium", "Large"]
).astype(str)

print("Clean rows:", len(clean_df))
print("Rejected rows:", len(rejected_df))
display(clean_df.head())
display(rejected_df[["TransactionReference", "TransactionDateText", "InstitutionCode", "CurrencyCode", "AmountText"]])

In [ ]:
# DataFrame aggregation and transformation.
currency_summary = (
    clean_df.groupby("CurrencyCode", as_index=False)
    .agg(
        TransactionCount=("TransactionReference", "count"),
        TotalAmount=("Amount", "sum"),
        TotalAmountLSL=("AmountLSL", "sum"),
        AverageAmount=("Amount", "mean"),
    )
    .round(2)
)

channel_pivot = pd.pivot_table(
    clean_df,
    values="AmountLSL",
    index="Channel",
    columns="CurrencyCode",
    aggfunc="sum",
    fill_value=0,
    margins=True,
).round(2)

display(currency_summary)
display(channel_pivot)

In [ ]:
# Public dataset example: World Bank inflation indicator.
# Source: World Bank API, indicator FP.CPI.TOTL.ZG (Inflation, consumer prices annual %).
def fetch_world_bank_indicator(indicator, countries=("LSO", "ZAF", "USA"), start_year=2020):
    country_part = ";".join(countries)
    url = f"https://api.worldbank.org/v2/country/{country_part}/indicator/{indicator}?format=json&per_page=20000"
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    payload = response.json()
    rows = payload[1] if len(payload) > 1 else []
    frame = pd.DataFrame([
        {
            "CountryCode": item["countryiso3code"],
            "Country": item["country"]["value"],
            "Year": int(item["date"]),
            "InflationPercent": item["value"],
        }
        for item in rows if item.get("value") is not None and int(item["date"]) >= start_year
    ])
    return frame.sort_values(["CountryCode", "Year"])

try:
    inflation_df = fetch_world_bank_indicator("FP.CPI.TOTL.ZG")
except Exception as exc:
    print("World Bank API unavailable, using a small fallback indicator table. Reason:", exc)
    inflation_df = pd.DataFrame({
        "CountryCode": ["LSO", "LSO", "ZAF", "ZAF"],
        "Country": ["Lesotho", "Lesotho", "South Africa", "South Africa"],
        "Year": [2024, 2025, 2024, 2025],
        "InflationPercent": [6.3, 5.8, 4.4, 4.1],
    })

display(inflation_df.tail(10))